### 사전 준비

In [1]:
import os,sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('utils'), '..')))
from module.utils import * 
from module.prompt import * 
from module.custom_model import *
from module.base_model import *
from module.tools import *

In [2]:
start_langsmith('development_1')

LangSmith 추적을 시작합니다.
[프로젝트명]
development_1


In [3]:
from typing import TypedDict, Annotated, List, Literal,Tuple
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage,ToolMessage
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks


### Text2SQL sub graphh process

In [4]:
# class SubState(TypedDict):
#     messages: Annotated[list, add_messages]

In [5]:
 
class SubState(TypedDict):
    question : Annotated[str,'user input question']   # 사용자 질의 or requeustion 질의
    plan : Annotated[list[str],'get plan_node']  # llm 생성한 작업 계획서
    messages : Annotated[list,add_messages]      # 작업 수행 후 얻은 데이터
    past_steps :Annotated[list, add_messages]
    answer : Annotated[str,' output final answer'] # 최종 답변 출력

In [6]:
# 오류 처리 함수
def handle_tool_error(state) -> dict:
    # 오류 정보 조회
    error = state.get("error")
    # 도구 정보 조회
    tool_calls = state["messages"][-1].tool_calls
    # ToolMessage 로 래핑 후 반환
    return {
        "messages": [
            ToolMessage(
                content=f"Here is the error: {repr(error)}\n\nPlease fix your mistakes.",
                tool_call_id=tc["id"],
            )
            for tc in tool_calls
        ]
    }

def tool_node_with_fallback(tools:list) -> RunnableWithFallbacks[Any, dict]:
    """
    Create a ToolNode with a fallback to handle errors and surface them to the agent.
    """
    # 오류 발생 시 대체 동작을 정의하여 ToolNode에 추가
    return ToolNode(tools).with_fallbacks(
        [RunnableLambda(handle_tool_error)], exception_key="error"
    )


# 쿼리 실행 부
@tool
def db_query_tool(query: str) -> str:
    """
    Run SQL queries against a database and return results
    Returns an error message if the query is incorrect
    If an error is returned, rewrite the query, check, and retry
    """
    # 쿼리 실행
    db = get_db()
    result = db.run_no_throw(query)

    # 오류: 결과가 없으면 오류 메시지 반환
    if "Error" in result:
        return f"Error: {result} \n\n . Please rewrite your query and try again."
    # 정상: 쿼리 실행 결과 반환
    elif not result:
        return "Success: value is None"
    else:
        return f"Success: {result}"



In [7]:
def instruction_node(state: SubState):
    plan = state['plan']
    plan_str = "\n".join(f"{i+1}. {text}" for i, text in enumerate(plan))
    task = plan[0]
    task_str = f"""For the following plan: \n\n {plan_str} \n\n You are tasked with executing [step 1. {task}]."""
    return SubState({'messages':[task_str]})

def get_table_list_node(state: SubState) -> dict[str, list[AIMessage]]:
    llm = get_gpt()
    tools = get_db_tool(llm)
    sql_db_list_tables = next(tool for tool in tools if tool.name == "sql_db_list_tables")
    llm_get_schema = llm.bind_tools([sql_db_list_tables],tool_choice='sql_db_list_tables')
    return SubState({ "messages": [llm_get_schema.invoke(state["messages"])]})

def get_all_table_node(state:SubState):
    llm = get_gpt()
    tools = get_db_tool(llm)
    sql_db_list_tables = next(tool for tool in tools if tool.name == "sql_db_list_tables")
    return tool_node_with_fallback([sql_db_list_tables])

def get_one_table_info_node(state:SubState):
    llm = get_gpt()
    tools = get_db_tool(llm)
    sql_db_schema = next(tool for tool in tools if tool.name == "sql_db_schema")
    llm_with_schema = llm.bind_tools([sql_db_schema],tool_choice='sql_db_schema')
    result = llm_with_schema.invoke(state['messages'])
    return SubState({'messages':[result]})

def get_one_table_schema_node(state:SubState):
    llm = get_gemini()
    tools = get_db_tool(llm)
    sql_db_schema = next(tool for tool in tools if tool.name == "sql_db_schema")
    return tool_node_with_fallback([sql_db_schema])

def get_query_gen_node(state:SubState):
    prompt = get_prompt_query_gen()
    llm =get_gpt()
    query_gen_llm = prompt | llm.bind_tools([db_query_tool],tool_choice='db_query_tool')
    history = state["messages"]
    query_gen =query_gen_llm.invoke({'placeholder':history})
    return SubState({ "messages": [query_gen]})

#  실패시 get_query_check_node -> execute_query 로직 추가
def get_query_check_node(state:SubState):
    prompt = get_prompt_query_check()
    llm = get_gpt().bind_tools([db_query_tool],tool_choice='db_query_tool')
    chain = prompt | llm 
    history = state["messages"]
    query_gen =chain.invoke({'placeholder':history})
    return SubState({ "messages": [query_gen]})

def execute_query(state:SubState):
    query = ''
    messages = state["messages"][-1]
    if len(messages.tool_calls) > 0:
        query = messages.tool_calls[0]['args']['query']
    else:
        query = state["messages"][-1].content
    response = db_query_tool(query)
    
    return SubState({'messages':ToolMessage(content=response,tool_call_id=state["messages"][-1].tool_calls[0]["id"])})

def answer_node(state:SubState):
    plan = state['plan']
    task = plan[0]
    prompt = get_prompt_query_gen()
    llm =get_gemini()
    query_gen_llm = prompt | llm
    history = state["messages"]
    result =query_gen_llm.invoke({'placeholder':history})
    print(result)
    latest_messages = f" Question :{task}\n Response :{result.content}"
    return SubState({'past_steps':[latest_messages],'messages':[latest_messages]})




In [ ]:
# routing 분기처리 수행 
def routing(state: SubState) -> Literal["get_query_check_node", "answer_node"]:
    latest_messages:str = state["messages"][-1].content
    if "Error:" in latest_messages:
        return "get_query_check_node"
    else:
        return "answer_node"

In [9]:
sub_state_graph = StateGraph(SubState)
sub_state_graph.add_node('instruction_node',instruction_node)
sub_state_graph.add_node('get_table_list_node',get_table_list_node)
sub_state_graph.add_node('get_all_table_node',get_all_table_node)
sub_state_graph.add_node('get_one_table_info_node',get_one_table_info_node)
sub_state_graph.add_node('get_one_table_schema_node',get_one_table_schema_node)
sub_state_graph.add_node("get_query_gen_node", get_query_gen_node)
sub_state_graph.add_node("get_query_check_node", get_query_check_node)

sub_state_graph.add_node("execute_query", execute_query)
sub_state_graph.add_node("answer_node", answer_node)

sub_state_graph.add_edge(START,'instruction_node')
sub_state_graph.add_edge('instruction_node','get_table_list_node')
sub_state_graph.add_edge('get_table_list_node','get_all_table_node')
sub_state_graph.add_edge('get_all_table_node','get_one_table_info_node')
sub_state_graph.add_edge('get_one_table_info_node','get_one_table_schema_node')

sub_state_graph.add_edge('get_one_table_schema_node','get_query_gen_node')

sub_state_graph.add_edge('get_query_gen_node','execute_query')
sub_state_graph.add_edge('get_query_check_node','execute_query')
sub_state_graph.add_conditional_edges(
    source='execute_query',
    path=routing
)

sub_state_graph.add_edge('answer_node',END)

sub_ck = get_check_pointer()
sub_graph = sub_state_graph.compile(checkpointer=sub_ck)

In [10]:
# inputs = {'messages':'Andrew Adam 직원의 인적정보를 모두 조회해줘'}
# sub_config = get_runnable_config(recursion_limit=10,thread_id=get_random_uuid())
# stream_graph(sub_graph,inputs,sub_config)

In [11]:
# sub_snapshot=sub_graph.get_state(sub_config)
# sub_snapshot

###  main 분기 처리 

In [12]:
class State(TypedDict):
    question : Annotated[str,'user input question']   # 사용자 질의 or requeustion 질의
    plan : Annotated[list[str],'get plan_node']  # llm 생성한 작업 계획서
    messages : Annotated[list,add_messages]      # 작업 수행 후 얻은 데이터
    past_steps :Annotated[list, add_messages]
    answer : Annotated[str,' output final answer'] # 최종 답변 출력

In [13]:
def retriever():
    loader = get_pdf_loader()
    splitter = get_text_splitter()
    docs = get_docs(loader,splitter)
    embedding = get_embedding()
    retrieve = get_retriever(docs,embedding)
    retrieve_tool = get_retriever_tool(retrieve)
    return retrieve_tool

def db_toolkit():
    db = SQLDatabase.from_uri("sqlite:///Chinook.db")
    toolkit = SQLDatabaseToolkit(db=db, llm=get_gpt())
    return toolkit.get_tools()



In [14]:
def plan_node(state:State) -> State:
    llm = get_gemini()
    prompt = get_prompt_music_planner()
    chain = prompt | llm.with_structured_output(MusicPlan)
    reponse = chain.invoke({'messages':[state['question']]})
    return State({'plan':reponse.steps}) 

def start_node(state:State):
    plan = state['plan']
    task = plan[0]
    prompt = get_prompt_start_node()   
    llm = get_gpt()
    chain = prompt | llm.with_structured_output(RouteModel) 
    result = chain.invoke({'placeholder':[task]})
    response = result.datasource
    return State({'messages':[HumanMessage(content=response)]})

def web_search_agent(state:State):
    plan = state['plan']
    plan_str = "\n".join(f"{i+1}. {text}" for i, text in enumerate(plan))
    task = plan[0]
    task_str = f"""For the following plan: \n\n {plan_str} \n\n You are tasked with executing [step 1. {task}]."""
    prompt = get_prompt_web()
    llm = get_gpt()
    tavily = get_tavily_tool()
    tools = [tavily,get_weather]
    web_agent = create_react_agent(model = llm,tools = tools,prompt=prompt)
    result = web_agent.invoke({'messages':task_str})
    latest_messages = f" Question :{task}\n Response :{result['messages'][-1].content}"
    return State({'past_steps':[latest_messages],'messages':[latest_messages]})

def pdf_loader_agent(state:State):
    prompt = get_prompt_multi_loader()
    llm = get_gpt()
    tool = retriever()
    tools = [tool]
    web_agent = create_react_agent(model = llm,tools = tools,prompt=prompt)
    result = web_agent.invoke({'messages':state['messages']})
    latest_messages = HumanMessage(
        content=result["messages"][-1].content, name="pdf_loader"
    )
    return State({'messages':[latest_messages]})

def conversation_agent(state:State):
    prompt = get_prompt_assistant()
    llm = get_gpt()
    chain = prompt | llm
    result = chain.invoke({'messages':state['messages']})
    return State({'messages':result})

def decision_node(state:State):
    prompt = get_prompt_replanner()
    llm = get_gpt().with_structured_output(MusicAct)
    chain = prompt | llm
    outputs =  chain.invoke({'input':state['question'],'plan':state['plan'],'past_steps':state['past_steps'][-1].content})
    # outputs => action=Plan(steps=['RAG의 작동 방식을 설명합니다.', 'RAG의 장단점을 설명합니다.', 'RAG의 활용 사례를 설명합니다.'])
    if isinstance(outputs.action,MusicResponse):
        return State({'answer':outputs.action.response})
    else:  # result == Plan 
        next_plan = outputs.action.steps
        if len(next_plan) == 0:
            return {"answer": "No more steps needed."}
        else:
            return {"plan": next_plan}
        
def final_generate_node(state: State):
    final_report = get_prompt_generate_markdown() | get_gemini() | StrOutputParser()
    answer = final_report.invoke({"input": state["question"], "past_steps": state['past_steps']})
    return {"answer": answer}

In [15]:
def is_routing_agent(state:State)->Literal['web_search_agent','pdf_loader_agent','sub_graph','conversation_agent']:
    datasource = state['messages'][-1].content
    if datasource=='web_search_agent':
        return 'web_search_agent'
    elif datasource == 'pdf_loader_agent':
        return 'pdf_loader_agent'
    elif datasource =='db_agent':
        return 'sub_graph'
    else :
        return 'conversation_agent'

def should_continue(state:State)->Literal['final_generate_node','start_node']:
    if "answer" in state and state["answer"]:
        return 'final_generate_node'
    else:
        return 'start_node'

In [16]:
state_graph = StateGraph(State)
state_graph.add_node('plan_node',plan_node)
state_graph.add_node('start_node',start_node)
state_graph.add_node('web_search_agent',web_search_agent)
state_graph.add_node('pdf_loader_agent',pdf_loader_agent)
state_graph.add_node('sub_graph',sub_graph)
state_graph.add_node('decision_node',decision_node)
state_graph.add_node('conversation_agent',conversation_agent)
state_graph.add_node('final_generate_node',final_generate_node)


state_graph.add_edge(START,'plan_node')
state_graph.add_edge('plan_node','start_node')
state_graph.add_conditional_edges(
    source='start_node',
    path=is_routing_agent
)


state_graph.add_edge('web_search_agent','decision_node')
state_graph.add_edge('pdf_loader_agent','decision_node')
state_graph.add_edge('sub_graph','decision_node')
state_graph.add_edge('conversation_agent','decision_node')
state_graph.add_conditional_edges(
    source='decision_node',
    path=should_continue
)
state_graph.add_edge('final_generate_node',END)


ck = get_check_pointer()
graph = state_graph.compile(checkpointer=ck)

In [17]:
# visualize_graph(graph)
# print(graph.get_graph().draw_mermaid())

### 
"[(1, 'Rock'), (2, 'Jazz'), (3, 'Metal'), (4, 'Alternative & Punk'), (5, 'Rock And Roll'), (6, 'Blues'), (7, 'Latin'), (8, 'Reggae'), (9, 'Pop'), (10, 'Soundtrack'), (11, 'Bossa Nova'), (12, 'Easy Listening'), (13, 'Heavy Metal'), (14, 'R&B/Soul'), (15, 'Electronica/Dance'), (16, 'World'), (17, 'Hip Hop/Rap'), (18, 'Science Fiction'), (19, 'TV Shows'), (20, 'Sci Fi & Fantasy'), (21, 'Drama'), (22, 'Comedy'), (23, 'Alternative'), (24, 'Classical'), (25, 'Opera')]"

In [20]:
inputs = {"question":'오늘 날씨에 어울리는 재즈 추천해줘, 내가 사는 곳은 서울이야'}
# inputs = {"question":'2025년 10월 2일 삼성전자 주식 종가'}
config = get_runnable_config(recursion_limit=10,thread_id=get_random_uuid())
invoke_graph(graph,inputs,config)


🔄 Node: plan_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
오늘 서울 날씨에 어울리는 재즈 음악 목록을 생성합니다.
생성된 음악 목록에서 서비스에서 이용 가능한 음악을 검증합니다.
최종 음악 목록을 선정하여 추천합니다.

🔄 Node: start_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================ Human Message =================================

web_search_agent

🔄 Node: agent in [web_search_agent] 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_hLko0iXAci46Zcs1mjF6SgwE)
 Call ID: call_hLko0iXAci46Zcs1mjF6SgwE
  Args:
    location: 서울

🔄 Node: tools in [web_search_agent] 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

Seoul,KR의 현재 날씨: overcast clouds, 온도: 24.18°C

🔄 Node: agent in [web_search_agent] 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message 

/tmp/ipykernel_429390/920615383.py:59: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = db_query_tool(query)



🔄 Node: get_query_gen_node in [sub_graph] 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  db_query_tool (call_PbiXxH2VweSfY5wM09nWI7gS)
 Call ID: call_PbiXxH2VweSfY5wM09nWI7gS
  Args:
    query: SELECT t.Name AS TrackName, a.Name AS ArtistName FROM Track t JOIN Album al ON t.AlbumId = al.AlbumId JOIN Artist a ON al.ArtistId = a.ArtistId WHERE t.Name IN ('Peace Piece', 'Blue in Green', 'My Funny Valentine', 'Naima', 'The Girl from Ipanema', 'The Look of Love', 'Don\'t Know Why');

🔄 Node: execute_query in [sub_graph] 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================

Error: Error: (sqlite3.OperationalError) near "t": syntax error
[SQL: SELECT t.Name AS TrackName, a.Name AS ArtistName FROM Track t JOIN Album al ON t.AlbumId = al.AlbumId JOIN Artist a ON al.ArtistId = a.ArtistId WHERE t.Name IN ('Peace Piec

GraphRecursionError: Recursion limit of 10 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/GRAPH_RECURSION_LIMIT

In [19]:
# snapshot = graph.get_state(config,subgraphs=True)
# snapshot.values